# Laboratorio 16. LSTM y GRU

**Pregunta guía:** ¿Cómo usan LSTM y GRU compuertas para modelar dependencias secuenciales?

## Objetivos
Al finalizar podrás preparar secuencias; entrenar LSTM y GRU; comparar con split fijo; inspeccionar errores y coste de parámetros.

## Prerrequisitos y conexión
Extiende la celda recurrente del laboratorio 15 con mecanismos de memoria controlada. Se requiere Python básico; cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
LSTM mantiene estado de celda: $c_t=f_t\odot c_{t-1}+i_t\odot\tilde c_t$ y estado oculto $h_t=o_t\odot\tanh(c_t)$. Las compuertas controlan olvido, escritura y salida. GRU mezcla estado previo y candidato con una compuerta de actualización; usa menos estados y parámetros.

## Problema y datos
Clasificamos secuencias sintéticas de dos frecuencias con los mismos splits. Todos los datos se generan en el notebook: no se requieren descargas ni archivos locales.

## Experimento conceptual
¿Qué diferencia observamos entre estas variantes en este problema y protocolo? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


PyTorch: 2.8.0+cpu | dispositivo: cpu


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Crear secuencias y partición
Generamos dos frecuencias con ruido y separamos train, validación y test.


In [2]:
samples, targets = [], []
for index in range(400):
    label = index % 2; time = np.linspace(0, 2*np.pi, 24)
    frequency = 1 if label == 0 else 3
    samples.append(np.sin(frequency*time + np.random.uniform(0, .3)) + np.random.normal(0, .15, len(time)))
    targets.append(label)
X = torch.tensor(np.array(samples), dtype=torch.float32, device=device).unsqueeze(-1)
y = torch.tensor(targets, dtype=torch.long, device=device)
order = torch.randperm(len(y), device=device); train_idx, val_idx, test_idx = order[:280], order[280:340], order[340:]
print(X.shape, '| tamaños:', len(train_idx), len(val_idx), len(test_idx))


torch.Size([400, 24, 1]) | tamaños: 280 60 60


**Resultado e interpretación.** Ambas arquitecturas reciben exactamente los mismos ejemplos y particiones. Las fases aleatorias y el ruido dificultan memorizar una única forma fija.


### Entrenar LSTM y GRU
Cambiamos únicamente el tipo de bloque recurrente y medimos test.


In [4]:
class SequenceClassifier(nn.Module):
    def __init__(self, recurrent_layer, name):
        super().__init__(); self.recurrent = recurrent_layer; self.name = name; self.output = nn.Linear(16, 2)
    def forward(self, inputs):
        result = self.recurrent(inputs)
        hidden = result[0][:, -1] if self.name == 'LSTM' else result[0][:, -1]
        return self.output(hidden)

In [5]:
scores = {}
for kind in ['LSTM', 'GRU']:
    recurrent = nn.LSTM(1, 16, batch_first=True) if kind == 'LSTM' else nn.GRU(1, 16, batch_first=True)
    model = SequenceClassifier(recurrent, kind).to(device); optimizer = torch.optim.Adam(model.parameters(), lr=.015)
    for epoch in range(12):
        loss = F.cross_entropy(model(X[train_idx]), y[train_idx]); optimizer.zero_grad(); loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad(): scores[kind] = (model(X[test_idx]).argmax(1) == y[test_idx]).float().mean().item()
print('exactitud test:', scores)


exactitud test: {'LSTM': 1.0, 'GRU': 1.0}


**Resultado e interpretación.** La comparación está limitada a una semilla y configuración. Conviene comparar varias semillas, parámetros y tiempo antes de preferir un bloque.


## Limitaciones y conclusiones clave
Los resultados dependen de esta semilla, tamaño de muestra, arquitectura y protocolo. No extrapoles métricas sintéticas a datos reales; separa decisiones de modelado de conclusiones generales. La celda de salida aporta la evidencia numérica y visual para responder la pregunta guía.

- Los parámetros y activaciones determinan cómo se representa la señal.
- La pérdida, el optimizador y el protocolo de evaluación forman parte del experimento.
- Una métrica aislada no describe todos los errores ni garantiza generalización.

## Ejercicios progresivos
1. **Guiado:** Guiado: inspecciona las formas de la salida y estado final de LSTM.
2. **Independiente:** Independiente: usa el mismo tamaño oculto y compara número de parámetros.
3. **Desafío:** Desafío: aumenta la longitud de secuencia sin cambiar el número de ejemplos y registra desempeño/tiempo.

## Conexión con el siguiente laboratorio
Usa la representación, operación o criterio de evaluación de este laboratorio como punto de partida del siguiente paso de la secuencia.
